In [1]:
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tmtools
from tmtools.io import get_residue_data, get_structure
import json

DRIVE_PATH = "/Users/samzirbo/Library/CloudStorage/GoogleDrive-sam.zirbo@gmail.com/My Drive/Masters/Semester 3/AlphaFold2 Ablation Study/04_Results/baseline"

PROTEIN = "LAT1"
PREDS_DIR = Path(DRIVE_PATH) / PROTEIN
REF_DIR = Path("../data") / PROTEIN / "references"
PAPER_CSV = Path("../external/af2_conformations/figures/fig1/fig1_data_models.csv")
METADATA = json.load(open(Path("../data") / "metadata.json"))

In [2]:
# Load reference structures once
IF_ID = METADATA[PROTEIN]["conformations"]["state_1"]["pdb_id"]
IF_CHAIN = METADATA[PROTEIN]["conformations"]["state_1"]["chain"]
IF_LABEL = METADATA[PROTEIN]["conformations"]["state_1"]["label"]
OF_ID = METADATA[PROTEIN]["conformations"]["state_2"]["pdb_id"]
OF_CHAIN = METADATA[PROTEIN]["conformations"]["state_2"]["chain"]
OF_LABEL = METADATA[PROTEIN]["conformations"]["state_2"]["label"]
ref_IF = get_structure(str(REF_DIR / f"{IF_LABEL}_{IF_ID}_{IF_CHAIN}.pdb"))
ref_OF = get_structure(str(REF_DIR / f"{OF_LABEL}_{OF_ID}_{OF_CHAIN}.pdb"))
ref_IF_coords, ref_IF_seq = get_residue_data(list(ref_IF.get_chains())[0])
ref_OF_coords, ref_OF_seq = get_residue_data(list(ref_OF.get_chains())[0])

# TM-score between the two references (used as the dotted crosshair)
ref_vs_ref = tmtools.tm_align(ref_IF_coords, ref_OF_coords, ref_IF_seq, ref_OF_seq)
print(f"IF vs OF TM-score: {ref_vs_ref.tm_norm_chain1:.5f}")
print("Paper uses:        0.86337")

IF vs OF TM-score: 0.86337
Paper uses:        0.86337


In [5]:
NSEQ = 5120  # max_msa depth used

rows = []
for pred_path in sorted(
    PREDS_DIR.glob(f"{PROTEIN}_unrelaxed_*_alphafold2_model_*_seed_*.pdb")
):
    pred = get_structure(str(pred_path))
    pred_coords, pred_seq = get_residue_data(list(pred.get_chains())[0])

    tm_if = tmtools.tm_align(
        pred_coords, ref_IF_coords, pred_seq, ref_IF_seq
    ).tm_norm_chain1
    tm_of = tmtools.tm_align(
        pred_coords, ref_OF_coords, pred_seq, ref_OF_seq
    ).tm_norm_chain1

    rows.append(
        {
            "protein": PROTEIN,
            "tm_IF": round(tm_if, 4),
            "tm_OF": round(tm_of, 4),
            "nseq": NSEQ,
        }
    )

ours = pd.DataFrame(rows)

# Save in the same format as the paper CSV
ours_csv = Path(f"../results/{PROTEIN}_tm_scores_eval_notebook.csv")
ours_csv.parent.mkdir(parents=True, exist_ok=True)
ours.to_csv(ours_csv, index=False)

print(f"Our predictions: {len(ours)} models → saved to {ours_csv}")
ours


Our predictions: 25 models → saved to ../results/LAT1_tm_scores_eval_notebook.csv


,protein,tm_IF,tm_OF,nseq
0,LAT1,0.9286,0.9299,5120
1,LAT1,0.9436,0.9049,5120
2,LAT1,0.9397,0.9137,5120
3,LAT1,0.9399,0.9094,5120
4,LAT1,0.9414,0.9060,5120
5,LAT1,0.9447,0.9050,5120
6,LAT1,0.9486,0.8964,5120
7,LAT1,0.9489,0.8945,5120
8,LAT1,0.9459,0.8987,5120
9,LAT1,0.9407,0.9024,5120
